# LLM-Powered Credit Decision Explanation with Feast

This notebook demonstrates how to combine **Feast feature store** with an **LLM** to:
1. Retrieve structured credit features from Feast
2. Generate a natural-language explanation of the credit decision
3. Use RAG (Retrieval Augmented Generation) with historical credit profiles as context

## Architecture
```
New Application
      │
      ▼
Feast Online Store (Redis)
      │  get_online_features()
      ▼
Structured Credit Profile
      │
      ▼
LLM (GPT / Ollama / HuggingFace)
      │  "Given this profile, explain the credit decision"
      ▼
Natural Language Explanation
```

## Supported LLM Backends
- **OpenAI** (GPT-4o-mini, GPT-4)
- **Ollama** (local: llama3, mistral)
- **HuggingFace Transformers** (local: Qwen2.5, Phi-3)

## Cell 1: Install Dependencies

In [ ]:
!pip install -q feast openai transformers pandas

## Cell 2: Configure LLM Backend

In [ ]:
import os

# ── Choose ONE backend ────────────────────────────────────────────────────────

# Option A: OpenAI (requires OPENAI_API_KEY env var)
LLM_BACKEND = "openai"
MODEL_NAME  = "gpt-4o-mini"
API_BASE    = None
API_KEY     = os.environ.get("OPENAI_API_KEY", "sk-...")

# Option B: Ollama (local, no API key needed)
# LLM_BACKEND = "ollama"
# MODEL_NAME  = "llama3"
# API_BASE    = "http://localhost:11434/v1"
# API_KEY     = "ollama"

# Option C: HuggingFace local model
# LLM_BACKEND = "huggingface"
# MODEL_NAME  = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Backend: {LLM_BACKEND}")
print(f"Model:   {MODEL_NAME}")

## Cell 3: Initialize Feast Feature Store

In [ ]:
from feast import FeatureStore

fs = FeatureStore(repo_path="../feature_repo")

feast_features = [
    "zipcode_features:city", "zipcode_features:state", "zipcode_features:location_type",
    "zipcode_features:tax_returns_filed", "zipcode_features:population",
    "zipcode_features:total_wages",
    "credit_history:credit_card_due", "credit_history:mortgage_due",
    "credit_history:student_loan_due", "credit_history:vehicle_loan_due",
    "credit_history:hard_pulls", "credit_history:missed_payments_2y",
    "credit_history:missed_payments_1y", "credit_history:missed_payments_6m",
    "credit_history:bankruptcies",
    "total_debt_calc:total_debt_due",
]

print("✅ Feast feature store initialized")
print(f"Project: {fs.project}")

## Cell 4: Retrieve Credit Features from Feast Online Store

In [ ]:
# Simulate an incoming loan application
application = {
    "zipcode":   76104,
    "dob_ssn":   "19630621_4278",
    "loan_amnt": 25000,
    "loan_intent": "DEBT_CONSOLIDATION",
    "person_home_ownership": "RENT",
}

# Fetch from Feast — same data the ML model used for the decision
result = fs.get_online_features(
    entity_rows=[{
        "zipcode":  application["zipcode"],
        "dob_ssn":  application["dob_ssn"],
        "loan_amnt": application["loan_amnt"],
    }],
    features=feast_features,
).to_dict()

# Flatten single-element lists
features = {k: v[0] if isinstance(v, list) else v for k, v in result.items()}

print("Credit features from Feast:")
for k, v in features.items():
    print(f"  {k:40s} {v}")

## Cell 5: Rule-Based Risk Scoring (Pre-LLM)

In [ ]:
def compute_risk_summary(features: dict) -> dict:
    """Simple rule-based risk score from Feast features."""
    score = 0
    flags = []

    if features.get("missed_payments_6m", 0) > 1:
        score += 30
        flags.append(f"Missed {features['missed_payments_6m']} payments in last 6 months")
    if features.get("bankruptcies", 0) > 0:
        score += 40
        flags.append(f"{features['bankruptcies']} bankruptcy on record")
    if features.get("hard_pulls", 0) > 3:
        score += 10
        flags.append(f"{features['hard_pulls']} recent credit inquiries (hard pulls)")
    if features.get("total_debt_due", 0) > 50000:
        score += 20
        flags.append(f"High total debt: ${features['total_debt_due']:,.0f}")

    level = "HIGH" if score >= 50 else "MEDIUM" if score >= 20 else "LOW"
    # Simulate ML model decision (in prod: use PyTorch or TF model output)
    decision = 0 if score >= 50 else 1
    return {"risk_score": min(score, 100), "risk_level": level, "flags": flags, "decision": decision}

risk = compute_risk_summary(features)
decision_label = "APPROVED" if risk["decision"] == 1 else "REJECTED"

print(f"Risk Score:  {risk['risk_score']}/100")
print(f"Risk Level:  {risk['risk_level']}")
print(f"Decision:    {decision_label}")
print(f"Flags:       {risk['flags']}")

## Cell 6: Format Credit Profile for LLM Prompt

In [ ]:
def format_credit_profile(features: dict, application: dict) -> str:
    return f"""Loan Application:
- Loan Amount:          ${application['loan_amnt']:,}
- Loan Intent:          {application['loan_intent']}
- Home Ownership:       {application['person_home_ownership']}

Credit History (from Feast):
- Total Debt Due:       ${features.get('total_debt_due', 0):,.0f}
- Credit Card Due:      ${features.get('credit_card_due', 0):,.0f}
- Mortgage Due:         ${features.get('mortgage_due', 0):,.0f}
- Student Loan Due:     ${features.get('student_loan_due', 0):,.0f}
- Vehicle Loan Due:     ${features.get('vehicle_loan_due', 0):,.0f}
- Hard Pulls (recent):  {features.get('hard_pulls', 0)}
- Missed Payments 6m:   {features.get('missed_payments_6m', 0)}
- Missed Payments 1y:   {features.get('missed_payments_1y', 0)}
- Missed Payments 2y:   {features.get('missed_payments_2y', 0)}
- Bankruptcies:         {features.get('bankruptcies', 0)}

Location Profile (from Feast):
- City / State:         {features.get('city', 'N/A')}, {features.get('state', 'N/A')}
- Population (zip):     {features.get('population', 0):,}
- Total Wages (zip):    ${features.get('total_wages', 0):,}"""

profile_text = format_credit_profile(features, application)
print(profile_text)

## Cell 7: Generate LLM Explanation

In [ ]:
SYSTEM_PROMPT = """You are a credit risk analyst assistant.
Given a loan applicant's financial profile retrieved from a feature store,
explain the credit decision clearly in 3-5 sentences.
Be factual, reference specific numbers, and avoid jargon."""

def generate_explanation(profile_text: str, decision_label: str, risk: dict) -> str:
    user_prompt = f"""The following loan application has been {decision_label}:

{profile_text}

Risk flags identified: {', '.join(risk['flags']) if risk['flags'] else 'None'}

Please explain in plain language why this application was {decision_label}."""

    if LLM_BACKEND in ("openai", "ollama"):
        from openai import OpenAI
        kwargs = {"api_key": API_KEY}
        if API_BASE:
            kwargs["base_url"] = API_BASE
        client = OpenAI(**kwargs)
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_prompt},
            ],
            temperature=0.3,
            max_tokens=512,
        )
        return response.choices[0].message.content.strip()

    elif LLM_BACKEND == "huggingface":
        from transformers import pipeline
        pipe = pipeline("text-generation", model=MODEL_NAME, max_new_tokens=512)
        prompt = f"{SYSTEM_PROMPT}\n\n{user_prompt}"
        output = pipe(prompt)
        return output[0]["generated_text"][len(prompt):].strip()

    return "LLM backend not configured."


explanation = generate_explanation(profile_text, decision_label, risk)

print("=" * 60)
print(f"CREDIT DECISION: {decision_label}")
print("=" * 60)
print()
print("LLM EXPLANATION:")
print(explanation)

## Cell 8: Batch Explanation for Multiple Applicants

In [ ]:
import pandas as pd

# Multiple applicants
applicants = [
    {"zipcode": 76104, "dob_ssn": "19630621_4278", "loan_amnt": 25000, "loan_intent": "DEBT_CONSOLIDATION", "person_home_ownership": "RENT"},
    {"zipcode": 10001, "dob_ssn": "19850315_1234", "loan_amnt": 10000, "loan_intent": "EDUCATION",          "person_home_ownership": "OWN"},
]

results = []
for app in applicants:
    result = fs.get_online_features(
        entity_rows=[{"zipcode": app["zipcode"], "dob_ssn": app["dob_ssn"], "loan_amnt": app["loan_amnt"]}],
        features=feast_features,
    ).to_dict()
    feats = {k: v[0] if isinstance(v, list) else v for k, v in result.items()}
    risk = compute_risk_summary(feats)
    d_label = "APPROVED" if risk["decision"] == 1 else "REJECTED"
    profile = format_credit_profile(feats, app)

    results.append({
        "dob_ssn":       app["dob_ssn"],
        "loan_amnt":     app["loan_amnt"],
        "decision":      d_label,
        "risk_score":    risk["risk_score"],
        "risk_level":    risk["risk_level"],
        "total_debt":    feats.get("total_debt_due", 0),
        "missed_6m":     feats.get("missed_payments_6m", 0),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

## Cell 9: Summary — Feast + LLM Pattern

| Component | Role |
|---|---|
| **Feast** | Single source of truth for credit features — training and serving |
| **`get_online_features`** | Low-latency feature retrieval at inference time |
| **Rule engine** | Fast decision (approve/reject) based on feature thresholds |
| **ML model** | Probabilistic scoring (PyTorch / TensorFlow) |
| **LLM** | Natural-language explanation of the decision for customer-facing output |
| **RAG** | Use similar historical profiles as context for the LLM |

### Why Feast matters here
The LLM explanation references real feature values pulled from Feast — the **same values** the ML model used to make the decision. Without a feature store, the explanation and the decision could reference different data, making the explanation misleading.